In [1]:
import pandas as pd

In [2]:
import numpy as np

In [3]:
weather_df = pd.read_csv(r"D:\weather_data.csv")

In [14]:
weather_df.dtypes

WindSpeedKmph             int64
WindDirDegree             int64
WeatherCode               int64
precipMM                float64
Visibilty                 int64
Pressure                  int64
Cloudcover                int64
DewPointF                 int64
WindGustKmph              int64
tempF                     int64
WindChillF                int64
Humidity                  int64
date             datetime64[ns]
time                     object
airport                  object
datetime         datetime64[ns]
dtype: object

In [4]:
flight_df = pd.read_csv(r"D:\combined_flight_data.csv")

In [5]:
flight_df = flight_df.rename(columns={'FlightDate': 'date'})

In [20]:
flight_df.dtypes

Year                        int64
Quarter                     int64
Month                       int64
DayofMonth                  int64
date               datetime64[ns]
Origin                     object
Dest                       object
CRSDepTime                 object
DepTime                   float64
DepDelayMinutes           float64
DepDel15                  float64
CRSArrTime                  int64
ArrTime                   float64
ArrDelayMinutes           float64
ArrDel15                  float64
CRSDepDatetime     datetime64[ns]
dtype: object

In [6]:
# Convert the 'date' column in both datasets to datetime for proper merging
weather_df['date'] = pd.to_datetime(weather_df['date'])
flight_df['date'] = pd.to_datetime(flight_df['date'])

In [7]:
# Convert 'time' in weather and 'CRSDepTime' in flight to comparable time formats (HH:MM)
# Handling cases where times are provided as integers like 700 (7:00 AM) or 1420 (2:20 PM)
weather_df['time'] = weather_df['time'].apply(lambda x: f"{int(x):04d}")
flight_df['CRSDepTime'] = flight_df['CRSDepTime'].apply(lambda x: f"{int(x):04d}")

In [8]:
# Convert the 'time' columns to datetime time format (HH:MM)
weather_df['time'] = pd.to_datetime(weather_df['time'], format='%H%M').dt.time
flight_df['CRSDepTime'] = pd.to_datetime(flight_df['CRSDepTime'], format='%H%M').dt.time

In [9]:
# Create a datetime column combining the date and time for both datasets
weather_df['datetime'] = pd.to_datetime(weather_df['date'].astype(str) + ' ' + weather_df['time'].astype(str))
flight_df['CRSDepDatetime'] = pd.to_datetime(flight_df['date'].astype(str) + ' ' + flight_df['CRSDepTime'].astype(str))

In [10]:
# Sorting is required for merge_asof to work correctly
flight_df = flight_df.sort_values('CRSDepDatetime')
weather_df = weather_df.sort_values('datetime')

In [30]:
merged_df = pd.merge_asof(flight_df, 
                          weather_df, 
                          left_on='CRSDepDatetime', 
                          right_on='datetime', 
                          left_by='Origin', 
                          right_by='airport', 
                          direction='backward')

In [32]:
# Drop redundant datetime columns and any unnecessary columns
merged_df = merged_df.drop(columns=['datetime', 'CRSDepDatetime'])

# Display the first few rows of the merged dataframe
print(merged_df.head())

# Save the merged dataframe to a new CSV file if needed
merged_df.to_csv(r"D:\merged_flight_weather.csv", index=False)


   Year  Quarter  Month  DayofMonth     date_x Origin Dest CRSDepTime  \
0  2016        1      1           1 2016-01-01    LAX  DFW   00:10:00   
1  2016        1      1           1 2016-01-01    SFO  CLT   00:15:00   
2  2016        1      1           1 2016-01-01    DEN  ATL   00:15:00   
3  2016        1      1           1 2016-01-01    PHX  CLT   00:15:00   
4  2016        1      1           1 2016-01-01    LAS  ATL   00:20:00   

   DepTime  DepDelayMinutes  ...  Pressure  Cloudcover  DewPointF  \
0     20.0             10.0  ...      1020           0         29   
1     11.0              0.0  ...      1023           0         33   
2      7.0              0.0  ...      1044           0          3   
3     12.0              0.0  ...      1019          31         26   
4     24.0              4.0  ...      1028           0         17   

   WindGustKmph  tempF  WindChillF  Humidity     date_y      time  airport  
0            17     59          58        39 2016-01-01  00:00:00    

In [34]:
merged_df

,Year,Quarter,Month,DayofMonth,date_x,Origin,Dest,CRSDepTime,DepTime,DepDelayMinutes,...,Pressure,Cloudcover,DewPointF,WindGustKmph,tempF,WindChillF,Humidity,date_y,time,airport
0,2016,1,1,1,2016-01-01,LAX,DFW,00:10:00,20.0,10.0,...,1020,0,29,17,59,58,39,2016-01-01,00:00:00,LAX
1,2016,1,1,1,2016-01-01,SFO,CLT,00:15:00,11.0,0.0,...,1023,0,33,23,41,35,71,2016-01-01,00:00:00,SFO
2,2016,1,1,1,2016-01-01,DEN,ATL,00:15:00,7.0,0.0,...,1044,0,3,21,7,-4,83,2016-01-01,00:00:00,DEN
3,2016,1,1,1,2016-01-01,PHX,CLT,00:15:00,12.0,0.0,...,1019,31,26,17,48,45,42,2016-01-01,00:00:00,PHX
4,2016,1,1,1,2016-01-01,LAS,ATL,00:20:00,24.0,4.0,...,1028,0,17,26,29,21,59,2016-01-01,00:00:00,LAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1877291,2017,4,12,31,2017-12-31,LAX,ATL,23:59:00,2358.0,0.0,...,1020,77,54,6,55,55,97,2017-12-31,23:00:00,LAX
1877292,2017,4,12,31,2017-12-31,LAX,DFW,23:59:00,143.0,104.0,...,1020,77,54,6,55,55,97,2017-12-31,23:00:00,LAX
1877293,2017,4,12,31,2017-12-31,LAX,ORD,23:59:00,824.0,505.0,...,1020,77,54,6,55,55,97,2017-12-31,23:00:00,LAX
1877294,2017,4,12,31,2017-12-31,LAS,ATL,23:59:00,2348.0,0.0,...,1021,11,11,8,50,48,20,2017-12-31,23:00:00,LAS
